# 04_2 — FAISS Vector Index (Chunk-Level)

Build a FAISS index over chunk embeddings for fast exact cosine similarity search.

**Why FAISS:** Pure local, no server needed, zero network latency. `IndexFlatIP` with
L2-normalized vectors gives exact cosine similarity. For < 100k chunks this is plenty fast.

**Inputs** (from notebooks 02 & 03):
- `data/processed/chunk_embeddings.npy` — (N_chunks, 384) float32
- `data/processed/chunks_with_embeddings.json` — chunk metadata aligned by row index
- `data/processed/resume_chunks.json` — full chunk text (for result previews)

**Outputs** (consumed by the Streamlit app):
- `resume_index.faiss` — FAISS index (DS3 member chunks only)
- `member_chunks_metadata.json` — chunk metadata aligned to the FAISS index
- `config.json` — model name, dimensions, counts

**Comparison with 04_1 (Qdrant):** FAISS is lower-level but faster for pure ANN.
Qdrant adds metadata filtering before search; FAISS filters post-hoc.

In [ ]:
import json
import numpy as np
from pathlib import Path
from collections import Counter
from sentence_transformers import SentenceTransformer
import faiss

PROJECT_ROOT  = Path("../").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

EMB_PATH      = PROCESSED_DIR / "chunk_embeddings.npy"
META_PATH     = PROCESSED_DIR / "chunks_with_embeddings.json"
CHUNKS_PATH   = PROCESSED_DIR / "resume_chunks.json"

# Final artifacts (paths match streamlit/config.py)
FAISS_OUT     = PROJECT_ROOT / "resume_index.faiss"
METADATA_OUT  = PROJECT_ROOT / "member_chunks_metadata.json"
CONFIG_OUT    = PROJECT_ROOT / "config.json"

MODEL_NAME    = "all-MiniLM-L6-v2"

print(f"Embeddings : {EMB_PATH}")
print(f"Chunk meta : {META_PATH}")
print(f"FAISS out  : {FAISS_OUT}")

## 1 — Load chunk embeddings & metadata

In [ ]:
all_embeddings = np.load(EMB_PATH)

with open(META_PATH, "r", encoding="utf-8") as f:
    all_meta = json.load(f)

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

assert len(all_embeddings) == len(all_meta) == len(all_chunks), (
    f"Row count mismatch: embeddings={len(all_embeddings)}, "
    f"meta={len(all_meta)}, chunks={len(all_chunks)}"
)

print(f"Loaded {len(all_embeddings)} chunk embeddings  (dim={all_embeddings.shape[1]})")

section_counts = Counter(m["section_type"] for m in all_meta)
print(f"\n{'Section':<20} {'Chunks':>8}")
print("-" * 30)
for sec, cnt in sorted(section_counts.items(), key=lambda x: -x[1]):
    print(f"{sec:<20} {cnt:>8}")

source_counts = Counter(m["source"] for m in all_meta)
print(f"\n{'Source':<20} {'Chunks':>8}")
print("-" * 30)
for src, cnt in sorted(source_counts.items(), key=lambda x: -x[1]):
    print(f"{src:<20} {cnt:>8}")

## 2 — Filter to DS3 member chunks (production index)

The production index only indexes DS3 member chunks — those are the candidates
being searched. Other sources (kaggle, train) are for training / evaluation only.

In [ ]:
member_indices = [i for i, m in enumerate(all_meta) if m["source"] in ("ds3_members", "ds3_board")]

member_embeddings = all_embeddings[member_indices].copy().astype("float32")
member_meta       = [all_meta[i] for i in member_indices]
member_chunks     = [all_chunks[i] for i in member_indices]

print(f"DS3 chunks selected for index: {len(member_indices)}")
sec_dist = Counter(m["section_type"] for m in member_meta)
for sec, cnt in sorted(sec_dist.items(), key=lambda x: -x[1]):
    print(f"  {sec:<20} {cnt}")

if len(member_indices) == 0:
    print("\n[WARNING] No DS3 chunks found — falling back to ALL chunks.")
    member_embeddings = all_embeddings.copy().astype("float32")
    member_meta       = all_meta
    member_chunks     = all_chunks

## 3 — Build the FAISS index

`IndexFlatIP` with L2-normalized vectors = exact cosine similarity search.
For < 100k chunks this runs in milliseconds and requires no training step.

In [ ]:
dimension = member_embeddings.shape[1]

# Vectors must already be L2-normalized (they are, from notebook 03).
# normalize_L2 is idempotent but harmless to call again as a safety net.
faiss.normalize_L2(member_embeddings)

index = faiss.IndexFlatIP(dimension)
index.add(member_embeddings)

print(f"FAISS index built")
print(f"  Index type : IndexFlatIP (exact cosine similarity)")
print(f"  Vectors    : {index.ntotal}")
print(f"  Dimension  : {dimension}")

## 4 — Test search

Chunk-level search returns individual evidence snippets. We then group the top
chunks by `candidate_id` and score candidates by their best-matching chunk
(max pooling) or average of top-k chunks per candidate.

In [ ]:
model = SentenceTransformer(MODEL_NAME)


def search_chunks(query: str, top_k: int = 20, section_filter: str | None = None):
    """
    Return the top-k most similar chunks to *query*.
    Optionally restrict to a specific section_type (e.g. 'experience', 'skills').
    """
    q_emb = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, top_k * 3)  # oversample before optional filter

    results = []
    for idx, score in zip(indices[0], scores[0]):
        if idx < 0:
            continue
        meta = member_meta[idx]
        if section_filter and meta["section_type"] != section_filter:
            continue
        results.append({
            "rank":         len(results) + 1,
            "score":        round(float(score), 4),
            "candidate_id": meta["candidate_id"],
            "section_type": meta["section_type"],
            "source":       meta["source"],
            "text":         member_chunks[idx]["text"][:300],
        })
        if len(results) >= top_k:
            break
    return results


def search_candidates(query: str, top_k_candidates: int = 10, chunks_per_candidate: int = 3):
    """
    Roll up chunk-level results to candidate-level using max-score pooling.
    Returns one row per candidate with their best chunk score and top matching snippets.
    """
    raw = search_chunks(query, top_k=top_k_candidates * chunks_per_candidate * 2)

    from collections import defaultdict
    candidate_hits: dict = defaultdict(list)
    for hit in raw:
        candidate_hits[hit["candidate_id"]].append(hit)

    candidates = []
    for cid, hits in candidate_hits.items():
        hits_sorted = sorted(hits, key=lambda h: -h["score"])
        candidates.append({
            "candidate_id": cid,
            "best_score":   hits_sorted[0]["score"],
            "top_chunks":   hits_sorted[:chunks_per_candidate],
        })

    candidates.sort(key=lambda c: -c["best_score"])
    return candidates[:top_k_candidates]


print("Search functions ready.")

In [ ]:
test_queries = [
    ("Python machine learning data science", None),
    ("React JavaScript full-stack web development", None),
    ("C++ computer vision embedded systems", None),
    ("built APIs with Python", "experience"),         # section-filtered search
    ("SQL database analytics", "skills"),             # skills-only search
]

for query, section_filter in test_queries:
    label = f"[{section_filter}] " if section_filter else ""
    print(f"\n{'=' * 70}")
    print(f"Query: {label}{query}")
    print(f"{'=' * 70}")
    results = search_chunks(query, top_k=5, section_filter=section_filter)
    for r in results:
        print(f"  #{r['rank']}  score={r['score']:.4f}  [{r['section_type']}]  {r['candidate_id']}")
        print(f"       {r['text'][:120]}")

## 5 — Candidate-level rollup

Show top candidates for a query by rolling up their best chunk scores.

In [ ]:
query = "Python machine learning data science model deployment"
print(f"Query: {query}\n")

candidates = search_candidates(query, top_k_candidates=5)
for c in candidates:
    print(f"  Candidate : {c['candidate_id']}  (best_score={c['best_score']:.4f})")
    for chunk in c["top_chunks"]:
        print(f"    [{chunk['section_type']}] score={chunk['score']:.4f}  {chunk['text'][:100]}")
    print()

## 6 — Export artifacts for Streamlit

In [ ]:
# 1. FAISS index
faiss.write_index(index, str(FAISS_OUT))
print(f"Saved FAISS index  : {FAISS_OUT}  ({FAISS_OUT.stat().st_size / 1024:.1f} KB)")

# 2. Chunk metadata — includes full text for UI previews
export_meta = []
for meta, chunk in zip(member_meta, member_chunks):
    export_meta.append({
        "chunk_id":     meta["chunk_id"],
        "candidate_id": meta["candidate_id"],
        "source":       meta["source"],
        "section_type": meta["section_type"],
        "text":         chunk["text"],
        "metadata":     meta.get("metadata", {}),
    })

with open(METADATA_OUT, "w", encoding="utf-8") as f:
    json.dump(export_meta, f, indent=2, ensure_ascii=False)
print(f"Saved chunk metadata: {METADATA_OUT}  ({METADATA_OUT.stat().st_size / 1024:.1f} KB)")

# 3. Config
config = {
    "model_name":          MODEL_NAME,
    "embedding_dimension": dimension,
    "num_chunks":          index.ntotal,
    "index_type":          "IndexFlatIP",
    "index_backend":       "faiss",
}
with open(CONFIG_OUT, "w") as f:
    json.dump(config, f, indent=2)
print(f"Saved config        : {CONFIG_OUT}")

print(f"\nDone! Streamlit app can load resume_index.faiss + member_chunks_metadata.json")